In [2]:
print("ram ram")

ram ram


In [3]:
# # Rule-based agentic pipeline generation for MES APIs

# This notebook shows a deterministic architecture for onboarding MES backend APIs into agentic pipelines.

# Principles:
# - Discover Swagger/OpenAPI APIs
# - Keep only APIs where metadata says `agent_accessible = true` and `auto_register = true`
# - Group by `business_domain`
# - Create one agent per domain
# - Build pipelines from ordered tool metadata
# - Use an LLM only for final reasoning over retrieved results, not for building the pipeline

In [4]:
import json
import requests
import pandas as pd
import urllib3
from requests.exceptions import ConnectionError, Timeout, RequestException

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://localhost:7204"
SWAGGER_URL = f"{BASE_URL}/swagger/v1/swagger.json"
TOKEN = None  # set this if your backend requires auth

headers = {"Accept": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"


def fetch_swagger(url, headers=None, verify=False):
    response = requests.get(url, headers=headers, verify=verify, timeout=15)
    response.raise_for_status()
    return response.json()

try:
    swagger = fetch_swagger(SWAGGER_URL, headers=headers)
    print("Swagger loaded successfully")
except Exception as e:
    print(f"Could not load swagger: {e}")
    swagger = {"paths": {}}

Could not load swagger: HTTPSConnectionPool(host='localhost', port=7204): Max retries exceeded with url: /swagger/v1/swagger.json (Caused by NewConnectionError("HTTPSConnection(host='localhost', port=7204): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))


In [ ]:
def parse_json_if_possible(value):
    if not value:
        return None
    if isinstance(value, dict) or isinstance(value, list):
        return value
    try:
        return json.loads(value)
    except Exception:
        return value

rows = []
for path, methods in swagger.get("paths", {}).items():
    for method, details in methods.items():
        if method.lower() not in {"get", "post", "put", "patch", "delete"}:
            continue

        metadata = {}
        description = parse_json_if_possible(details.get("description"))
        if isinstance(description, dict):
            metadata.update(description)
        elif isinstance(description, str):
            metadata["description_text"] = description

        ext = details.get("x-metadata", {})
        if isinstance(ext, dict):
            metadata.update(ext)

        tool_name = (
            metadata.get("tool_name")
            or details.get("operationId")
            or f"{method.upper()}_{path.replace('/', '_')}"
        )

        rows.append({
            "tool_name": tool_name,
            "path": path,
            "method": method.upper(),
            "description": metadata.get("description") or details.get("summary") or "",
            "business_domain": metadata.get("business_domain") or "Unassigned",
            "recommended_agents": metadata.get("recommended_agents", []),
            "restricted_agents": metadata.get("restricted_agents", []),
            "agent_accessible": bool(metadata.get("agent_accessible", True)),
            "auto_register": bool(metadata.get("auto_register", True)),
            "operation_type": metadata.get("operation_type") or method.upper(),
            "risk_level": metadata.get("risk_level") or "MEDIUM",
            "priority": int(metadata.get("priority", 100)),
        })

backend_api_df = pd.DataFrame(rows)
backend_api_df.head()

c:\Users\renuk\OneDrive\Desktop\denso code\super_agent_backend\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
usable_api_df = backend_api_df[
    (backend_api_df["agent_accessible"] == True) &
    (backend_api_df["auto_register"] == True)
].copy()

usable_api_df = usable_api_df.sort_values(["business_domain", "priority", "tool_name"]).reset_index(drop=True)
print(f"Total discovered APIs: {len(backend_api_df)}")
print(f"Usable APIs for agentic pipelines: {len(usable_api_df)}")

usable_api_df[["tool_name", "business_domain", "operation_type", "priority", "agent_accessible", "auto_register"]].head(20)

{'tool_name': 'get_work_orders',
 'description': 'Returns active work orders for AI-driven production monitoring and scheduling. Accessible by the Agentic AI platform for automatic tool registration and execution. Recommended for Operations and Production Planning agents only. Not intended for maintenance, inventory, or finance-related tasks.',
 'agent_accessible': True,
 'auto_register': True}

In [ ]:
def build_agent_definitions(api_df):
    agents = []
    for domain, group in api_df.groupby("business_domain", sort=True):
        if group.empty:
            continue
        agents.append({
            "agent_name": f"{domain} Agent",
            "business_domain": domain,
            "tool_count": len(group),
            "tools": group["tool_name"].tolist(),
            "registration_mode": "rule_based",
        })
    return pd.DataFrame(agents)

agent_registry_df = build_agent_definitions(usable_api_df)
agent_registry_df

In [ ]:
def build_rule_based_pipeline_registry(api_df):
    pipeline_rows = []
    pipeline_graph = {}

    for domain, group in api_df.groupby("business_domain", sort=True):
        if group.empty:
            continue

        ordered_group = group.sort_values(["priority", "tool_name"], ascending=[True, True]).reset_index(drop=True)
        selected_tools = ordered_group.head(5).copy()

        agent_name = f"{domain} Agent"
        pipeline_name = f"{domain.lower().replace(' ', '_')}_pipeline"

        graph_steps = []
        for idx, row in selected_tools.iterrows():
            step_name = f"step_{idx + 1}"
            step_id = f"{pipeline_name}_{step_name}"
            depends_on = [] if idx == 0 else [f"{pipeline_name}_step_{idx}"]
            graph_steps.append({
                "step_id": step_id,
                "step_name": step_name,
                "tool_name": row["tool_name"],
                "operation_type": row["operation_type"],
                "priority": int(row["priority"]),
                "depends_on": depends_on,
            })

            pipeline_rows.append({
                "pipeline_name": pipeline_name,
                "agent_name": agent_name,
                "business_domain": domain,
                "step_order": idx + 1,
                "step_name": step_name,
                "tool_name": row["tool_name"],
                "path": row["path"],
                "operation_type": row["operation_type"],
                "priority": int(row["priority"]),
                "uses_llm": False,
                "reason": "metadata_ordered_rule_based",
            })

        pipeline_graph[agent_name] = {
            "business_domain": domain,
            "pipeline_name": pipeline_name,
            "steps": graph_steps,
        }

    return pd.DataFrame(pipeline_rows), pipeline_graph


pipeline_registry_df, pipeline_graph = build_rule_based_pipeline_registry(usable_api_df)
pipeline_registry_df.head(15)

In [ ]:
def build_runtime_execution_plan(question, api_df, top_k=5):
    selected_tools = select_tools_for_query(question, api_df, top_k=top_k)
    if not selected_tools:
        return {
            "domain": "Unassigned",
            "agent_name": "Unassigned Agent",
            "selected_tools": [],
            "llm_required": True,
            "reason": "no_metadata_match",
        }

    domain = selected_tools[0].get("business_domain", "Unassigned")
    return {
        "domain": domain,
        "agent_name": f"{domain} Agent",
        "selected_tools": [tool.get("tool_name") for tool in selected_tools],
        "llm_required": True,
        "reason": "execute_selected_tools_then_reason",
    }


example_runtime_plan = build_runtime_execution_plan(example_question, usable_api_df)
example_runtime_plan

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# Optional: keep the database connection setup for later persistence
# of generated agents and pipelines into the backend tables.


In [36]:
DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "manufacturing_ai",
    "user": "postgres",
    "password": "0987654321"
}

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

engine = create_engine(DATABASE_URL)

In [37]:
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version();"))
        print(result.fetchone()[0])
        print("\n✅ PostgreSQL Connected Successfully")
except Exception as e:
    print(e)

PostgreSQL 16.14, compiled by Visual C++ build 1944, 64-bit

✅ PostgreSQL Connected Successfully


In [38]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
ORDER BY table_name;
"""

tables = pd.read_sql(query, engine)
tables

,table_name
0,agent_execution_logs
1,agent_tool_mapping
2,agentic_pipline
3,alert_master
4,api_logs
5,app_authentication
6,app_connection_table
7,app_table
8,app_version_history
9,approval_history


In [39]:
query = """
SELECT
table_name,
column_name,
data_type
FROM information_schema.columns
WHERE table_schema='public'
ORDER BY table_name, ordinal_position;
"""

columns = pd.read_sql(query, engine)
columns

,table_name,column_name,data_type
0,agent_execution_logs,execution_id,uuid
1,agent_execution_logs,agentic_id,character varying
2,agent_execution_logs,started_at,timestamp with time zone
3,agent_execution_logs,completed_at,timestamp with time zone
4,agent_execution_logs,status,character varying
...,...,...,...
173,workflow_history,input_state,jsonb
174,workflow_history,output_state,jsonb
175,workflow_history,approval_status,character varying
176,workflow_history,customer_id,integer


In [40]:
query = """
SELECT COUNT(*)
FROM information_schema.tables
WHERE table_schema='public';
"""

pd.read_sql(query, engine)

,count
0,26


In [ ]:
def select_tools_for_query(question, api_df, top_k=5):
    """Rule-based routing: use metadata keywords to pick the most relevant APIs."""
    q = (question or "").lower()
    scored = []
    for _, row in api_df.iterrows():
        text = " ".join([
            str(row.get("tool_name", "")),
            str(row.get("description", "")),
            str(row.get("business_domain", "")),
            str(row.get("operation_type", "")),
        ]).lower()
        score = 0
        if row.get("business_domain", "") and row.get("business_domain", "").lower() in q:
            score += 3
        for keyword in ["work", "order", "machine", "production", "inventory", "quality", "maintenance", "dispatch", "warehouse", "shift", "status"]:
            if keyword in q and keyword in text:
                score += 2
        if row.get("operation_type", "").lower() == "read" and "status" in q:
            score += 1
        scored.append((score, row))

    scored.sort(key=lambda x: x[0], reverse=True)
    selected = [row for _, row in scored[:top_k] if _[0] > 0]
    if not selected:
        selected = [row for _, row in scored[:top_k]]
    return selected


example_question = "Which machine is overloaded today?"
selected_tools = select_tools_for_query(example_question, usable_api_df)

print("Question:", example_question)
print("Selected tools:")
for tool in selected_tools:
    print(f"- {tool['tool_name']} [{tool['business_domain']}]")

# Only use LLM after the tool set is selected and results are gathered.
# This keeps the LLM call minimal and focused on reasoning over retrieved data.
